In [ ]:
import sys
from pathlib import Path

# Set root
cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.evaluation import splits
from src.evaluation.make_splits import load_split

# Paths
DATA_PATH = ROOT / "notebooks" / "merge" / "groundwater_ml_integration_lag1_2021_2024_with_climate.csv"
FEATURE_SEL = ROOT / "results" / "feature_selection.csv"

In [2]:
# Load the cleaned/merged dataset

df = pd.read_csv(DATA_PATH, dtype={"gm_well_id": "string"}, parse_dates=["gm_samp_collection_date"])
df = load_split(df)

feature_selection = pd.read_csv(FEATURE_SEL)

print(df["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(df["holdout"].sum()))

{0: 1035, 1: 972, 2: 1143, 3: 1041, 4: 1089} | holdout rows: 1089


In [20]:
# Set the training folds and the holdout

train_val = df[~df["holdout"]].reset_index(drop=True)       # Training set with 4 folds

TARGET = "nitrate_mg_L"
FEATURES = []          # Fill in with features used
X = train_val[FEATURES]
y = np.log1p(train_val[TARGET])

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 1035, 1: 972, 2: 1143, 3: 1041}


In [23]:
my_feats = feature_selection.loc[feature_selection["decision"]=="keep", "feature"].tolist()
my_feats

df.head()

,gm_well_id,gm_well_category,gm_samp_collection_date,gm_latitude,gm_longitude,nitrate_mg_L,year,source_csv_row,observation_id,soil_mapped_coverage_pct,...,crop_map_reference_year,historical_mean_annual_precip_mm,previous_3_complete_months_precip_mm,previous_12_complete_months_precip_mm,historical_mean_temperature_c,previous_3_complete_months_temperature_c,previous_12_complete_months_temperature_c,cell,fold,holdout
0,AGL020000465-PRIMARY,Domestic,2022-05-04 09:40:00,35.551290,-120.600420,3.12,2022,2,GW-CSV-00002,100.0,...,NaN,347.528444,37.846,282.194,15.820988,11.796296,15.629630,-6_-28,2,False
1,AGL020000465-PRIMARY,Domestic,2023-06-07 11:45:00,35.551290,-120.600420,2.97,2023,3,GW-CSV-00003,100.0,...,NaN,338.429600,232.664,838.200,15.817593,13.037037,15.268519,-6_-28,2,False
2,AGL020000465-PRIMARY,Domestic,2024-05-16 08:00:00,35.551290,-120.600420,3.38,2024,4,GW-CSV-00004,100.0,...,2023.0,377.420909,312.420,538.226,15.727273,11.351852,15.134259,-6_-28,2,False
3,AGL020000465-PRIMARY,Domestic,2025-04-14 08:00:00,35.551290,-120.600420,3.14,2025,5,GW-CSV-00005,100.0,...,2023.0,384.429000,169.164,299.720,15.709877,9.962963,15.467593,-6_-28,2,False
4,AGL020000484-ROOS-HOMESTEAD,Domestic,2022-05-18 12:50:00,35.515854,-120.640089,3.82,2022,6,GW-CSV-00006,100.0,...,NaN,417.406667,39.370,307.848,15.368313,11.425926,15.314815,-6_-28,2,False


Here you can begin training your model.

In [6]:
# # Train your models here

# for train, test in splits.iter_folds(train_val["fold"]):
#     model =            # Your model here
#     model.fit(X[train], y[train])
#     # Fill in the rest
    

Save this last file for the final evaluation (when we have compared all of our models and have chosen a final one).

In [7]:
# # ============================================================
# # FINAL EVALUATION ON THE HOLD-OUT FOLD
# # Run this ONCE, for the final chosen model only, after all
# # model choices (features, hyperparameters) are finished.
# # ============================================================
# RUN_FINAL_EVALUATION = False

# if RUN_FINAL_EVALUATION:
#     holdout_df = df[df["holdout"]].reset_index(drop=True)
#     final_model = ...                      # the chosen model, refit on all of train_val
#     final_model.fit(X, y)
#     holdout_pred = final_model.predict(holdout_df[FEATURES])
#     ...                                    # scores on np.log1p(holdout_df[TARGET])
